<a href="https://colab.research.google.com/github/Yahimeen/IA_AplicadaLLama_Challenge_02_02/blob/main/Hands_On_Dise%C3%B1o_de_Agentes_Conversacionales_Challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: DISEÑO DE AGENTES CONVERSACIONALES**

Una vez vista la masterclass ***Diseño de Agentes Conversacionales***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.


De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/18wkM2LSfY39VZ2GBdcPCY2Qw0EzewJsx?usp=drive_link).

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**


#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Leer credenciales desde Colab Secrets
from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

print("Credenciales cargadas correctamente.")

#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [ ]:
# Instalar Ollama

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [ ]:
# Instalar dependencias
!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [ ]:
# Configurar el servidor del webhook
from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()

In [ ]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que guarda, por cada número de teléfono, el historial reciente de mensajes — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación. Esta es una implementación manual y simplificada del concepto; la lectura del Tema menciona **Llama Stack** como el framework que estandariza esta pieza (junto con herramientas y seguridad) para no reconstruirla en cada proyecto. **Nota:** el patrón exacto de integración de Llama Stack debe confirmarse contra su documentación vigente antes de grabar o dar la sesión en vivo.

In [ ]:
# Una conversacion de Llama Stack por numero de telefono
conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **GENERAR LA RESPUESTA CON HISTORIAL Y UNA REGLA FIJA**

Se define la lógica que arma cada respuesta: si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento (una decisión que conviene fijar por regla, no dejarla al modelo); en cualquier otro caso, se reenvía todo el historial reciente a Llama, para que pueda resolver referencias a mensajes anteriores.

In [ ]:
# Definir la funcion de generacion de respuesta con memoria

FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "agente humano"]

SYSTEM_PROMPT = (
    "Eres el asistente de agendado de citas de una clinica dental. "
    "Responde de forma breve (2-3 lineas)."
)

def generar_respuesta(numero, texto_usuario):
    texto_low = texto_usuario.lower()
    if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
        return "Claro, en un momento un miembro del equipo te contacta directamente."

    conversation_id = obtener_conversacion(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
    )
    return respuesta.output_text

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [ ]:
# Definir endpoint de verificacion de credenciales
@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}

In [ ]:
# Definir endpoint de recepcion de mensajes
@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}

In [ ]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v26.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [ ]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama
errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(numero, "Claro, en un momento un miembro del equipo te contacta directamente.")
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)
        except Exception as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "error": repr(e)})

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [ ]:
# Instalar e iniciar tunel publico con ngrok
!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading
import time

# Cerrar túneles viejos de esta sesión, si existen
try:
    ngrok.kill()
except Exception:
    pass

# Si el servidor se ejecutó antes en este runtime, pedirle que termine
try:
    servidor.should_exit = True
    time.sleep(1)
except Exception:
    pass

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info",
)
servidor = uvicorn.Server(config)

def correr_servidor():
    servidor.run()

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

# Dar tiempo a Uvicorn para abrir el puerto
time.sleep(2)

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000, "http")

CALLBACK_URL = tunel_publico.public_url + "/webhook"

print("Servidor iniciado.")
print("\nConfigura en Meta:")
print("Callback URL:", CALLBACK_URL)
print("Verify Token: usa el valor del secreto WHATSAPP_VERIFY_TOKEN")
print("\nDespués suscríbete al campo 'messages' del webhook.")

### **PROBAR EL FLUJO COMPLETO**

Se manda un primer mensaje mencionando el servicio dental, y un segundo mensaje que depende del primero — para confirmar que el historial sí se está reenviando.

In [ ]:
# Mandar el primer mensaje, mencionando el servicio

payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "12033002267", "text": {"body": "Quiero una cita dental"}} # 5215500000000
    ]}}]}]
}

r1 = requests.post("http://0.0.0.0:8000/webhook", json=payload_1)
print("Mensaje 1:", r1.status_code, r1.text)


In [ ]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio
payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "12033002267", "text": {"body": "¿Tienen horario el sábado?"}}
    ]}}]}]
}

r2 = requests.post("http://0.0.0.0:8000/webhook", json=payload_2)
print("Mensaje 2:", r2.status_code, r2.text)



In [ ]:
# Verificar respuestas
respuestas_enviadas

In [ ]:
# Ver mensajes recibidos
mensajes_recibidos

In [ ]:
# Verificar almacenamiento en Llama Stock
numero = normalizar_numero_mx("525626131742")  # tu numero real de WhatsApp
conv_id = conversaciones_por_numero[numero]

items = client.conversations.items.list(conversation_id=conv_id)
for item in items.data:
    print(item.role, "->", item.content)

In [ ]:
# Finalizar proceso de tunelización

ngrok.kill()
print('Proceso de ngrok cerrado.')

## **CHALLENGE: AGENTE DE UNA VETERINARIA**

Una vez visto el ***Hands-On: Diseño de Agentes Conversacionales***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un agente similar, pero para una veterinaria: además de la regla de escalamiento a una persona, se agrega una regla de urgencia médica, y un límite al historial que se conserva en la conversación de Llama Stack.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Prepara el entorno:** Instala y arranca Ollama en Colab, descarga el modelo Llama 3.1 8B, instala y levanta **Llama Stack** con Ollama registrado, y carga las credenciales de WhatsApp desde **Colab Secrets**.

In [ ]:
# Instalar Ollama (requiere zstd, que Colab no trae por defecto)

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Iniciar el servidor de Ollama en segundo plano
import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Ollama listo.")

In [ ]:
# Instalar dependencias
!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

In [ ]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

In [ ]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

print("Credenciales cargadas correctamente.")

**2. Construye la memoria de conversación:** Crea una conversación de **Llama Stack** por número de teléfono (`client.conversations.create()`) con una memoria de respuesta.

In [ ]:
# Conversacion de Llama Stack por numero de telefono
conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

In [ ]:
# Definir la funcion de generacion de respuesta con memoria

FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "agente humano", "urgencia medica", "cita urgente"]

SYSTEM_PROMPT = (
    "Eres el asistente de manejo de agendado de citas de una clinica veterinaria. "
    "Responde de forma breve (2-3 lineas)."
)

def generar_respuesta(numero, texto_usuario):
    texto_low = texto_usuario.lower()
    if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
        return "Claro, en un momento un miembro del equipo te contacta directamente."

    conversation_id = obtener_conversacion(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
    )
    return respuesta.output_text

**3. Agrega la regla fija y protege el procesamiento por número:**

   * Cuando el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento.
   * Igual que en el Hands-On, usa un lock por número de teléfono para que dos mensajes seguidos del mismo usuario no disparen dos llamadas en paralelo sobre la misma conversación.

In [ ]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

In [ ]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama

errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(numero, "Claro, en un momento un miembro del equipo te contacta directamente.")
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)
        except Exception as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "error": repr(e)})

**4. Define los endpoints del webhook:** usa `BackgroundTasks` para responder de inmediato a Meta, delegando el procesamiento real a la función anterior.

In [ ]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()

In [ ]:
# Definir función de normalización de número
def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

In [ ]:
# Definir endpoint de verificacion de credenciales
@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}

In [ ]:
# Definir endpoint de recepcion de mensajes
@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}

In [ ]:
# Definir funcion de envio de respuesta
respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v26.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

**5. Expón el servidor con ngrok y prueba el flujo completo:**

   * Manda una conversación de al menos 4 mensajes y confirma que el historial se recorta correctamente.
   * Manda un mensaje con una palabra de urgencia y confirma que dispara la regla fija en vez de una respuesta de Llama.

In [ ]:
# Instalar e iniciar tunel publico con ngrok
!pip install pyngrok --quiet

from pyngrok import ngrok
import uvicorn
import threading
import time

# Cerrar túneles viejos de esta sesión, si existen
try:
    ngrok.kill()
except Exception:
    pass

# Si el servidor se ejecutó antes en este runtime, pedirle que termine
try:
    servidor.should_exit = True
    time.sleep(1)
except Exception:
    pass

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    log_level="info",
)
servidor = uvicorn.Server(config)

def correr_servidor():
    servidor.run()

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()

# Dar tiempo a Uvicorn para abrir el puerto
time.sleep(2)

ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
tunel_publico = ngrok.connect(8000, "http")

CALLBACK_URL = tunel_publico.public_url + "/webhook"

print("Servidor iniciado.")
print("\nConfigura en Meta:")
print("Callback URL:", CALLBACK_URL)
print("Verify Token: usa el valor del secreto WHATSAPP_VERIFY_TOKEN")
print("\nDespués suscríbete al campo 'messages' del webhook.")


In [ ]:
# Mandar el primer mensaje, mencionando el servicio
payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "12033002267", "text": {"body": "Quiero una cita dental"}} # 5215500000000
    ]}}]}]
}

r1 = requests.post("http://0.0.0.0:8000/webhook", json=payload_1)
print("Mensaje 1:", r1.status_code, r1.text)

In [ ]:
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio

payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "12033002267", "text": {"body": "¿Tienen horario el sábado?"}}
    ]}}]}]
}

r2 = requests.post("http://0.0.0.0:8000/webhook", json=payload_2)
print("Mensaje 2:", r2.status_code, r2.text)


In [ ]:
# Verificar respuestas
respuestas_enviadas

In [ ]:
# Ver mensajes recibidos
mensajes_recibidos

In [ ]:
# Verificar almacenamiento en Llama Stack

numero = normalizar_numero_mx("12033002267")  # tu numero real de WhatsApp
conv_id = conversaciones_por_numero[numero]

items = client.conversations.items.list(conversation_id=conv_id)
for item in items.data:
    print(item.role, "->", item.content)

In [ ]:
# Finalizar proceso de tunelización
ngrok.kill()
print('Proceso de ngrok cerrado.')